# Paper Figures: Figure S2 group similarities with SIMBA data

This notebook generates publication-ready figures for the Bazzino & Roitman sodium appetite manuscript using data assembled by `src/assemble_all_data.py`.

**Figure S2: Grouped (SIMBA) Analysis** — Comparing different groups.

In [ ]:
%load_ext IPython.extensions.autoreload
%autoreload 2

import pathlib
from pathlib import Path
import sys
# Add src to path for importing local modules
sys.path.insert(0, str(Path("../src").resolve()))
from pickle_compat import enable_dill_pathlib_compat
enable_dill_pathlib_compat()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import dill
from scipy import stats
from scipy.ndimage import gaussian_filter1d
from scipy.spatial.distance import cdist
from sklearn.manifold import MDS
from trompy import save_figure_atomic
from extract_behav_parameters import smooth_array


from figure_config import (
    configure_matplotlib, COLORS, HEATMAP_CMAP_DIV,
    HEATMAP_CMAP_RED, HEATMAP_CMAP_BLUE,
    DATAFOLDER, RESULTSFOLDER, FIGSFOLDER,
    HEATMAP_VLIM_BEHAV, YLIMS_BEHAV,
    BEHAV_SMOOTH_WINDOW, SAVE_FIGS
)
from figure_plotting import (
    smooth_array, get_heatmap_data_by_rat, get_mean_snips, get_auc,
    init_heatmap_figure, init_snips_figure, make_heatmap,
    plot_snips, plot_auc_summary, print_auc_stats, get_trial_data_by_rat,
    scale_vlim_to_data, calculate_ylims,
    draw_regression_line, make_correlation_plot_simba
)

# Configure matplotlib
configure_matplotlib()
colors = COLORS  # Use shared color palette
custom_cmap_red = HEATMAP_CMAP_RED  # Use shared colormap
custom_cmap_blue = HEATMAP_CMAP_BLUE  # Use shared colormap

custom_cmap = HEATMAP_CMAP_DIV  # Use shared diverging colormap

SAVE_FIGS = True

## Load Assembled Data

Load the complete dataset from the pickle file generated by the assembly script.

In [ ]:
assembled_data_path = DATAFOLDER / "assembled_data.pickle"

with open(assembled_data_path, "rb") as f:
    data = dill.load(f)

# Extract main components
x_array = data["x_array"]
snips_simba = data["snips_simba"]

params = data.get("params", {})
metadata = data.get("metadata", {})

print(f"Loaded assembled data from {assembled_data_path}")
print(f"\nData structure:")
print(f"  - x_array shape: {x_array.shape}")
print(f"  - snips_simba shape: {snips_simba.shape}")
print(f"  - x_array columns: {x_array.columns.tolist()}")
print(f"  - Number of trials: {len(x_array)}")


## Figure S2: Heatmaps of individual rats per group

In [ ]:
snips_to_plot = np.array(smooth_array(snips_simba))  # Use smoothed and z-scored data for all subsequent analyses and plotting

# Parameters for visualization - MOVEMENT
# Use dynamic scaling based on actual data ranges (asymmetric, not symmetric)
# This ensures heatmaps use the full range of your data without artificial symmetry
vmin = np.nanpercentile(snips_to_plot, 5)  # 5th percentile for lower bound
vmax = np.nanpercentile(snips_to_plot, 95)  # 95th percentile for upper bound
vlim = (vmin, vmax)
print(f"Calculated asymmetric vlims: vmin={vmin:.4f}, vmax={vmax:.4f}")
# vlim = (-0.75, 0.75) # for zscored data
vlim = (-1, 1)

# Calculate dynamic y-limits based on all snips data - MOVEMENT
print("\n" + "="*60)
print("CALCULATING Y-LIMITS FOR TIME SERIES PLOTS")
print("="*60)

# Get all condition/infusion combinations to compute limits
rep_10, rep_45 = get_mean_snips(snips_simba, x_array, "replete")
dep_10, dep_45 = get_mean_snips(snips_simba, x_array, "deplete")

# Calculate y-limits based on all snips with 5% padding
calc_ylims = calculate_ylims([rep_10, rep_45, dep_10, dep_45], pad_percentage=5)

print(f"  Data range (min, max): ({np.nanmin(snips_simba):.4f}, {np.nanmax(snips_simba):.4f})")
print(f"  Calculated y-limits: {calc_ylims}")
print("="*60 + "\n")

# Use calculated limits or override with fixed values if preferred
ylims = calc_ylims
ylims = (0, 8)


In [ ]:
### 2A-D Heatmaps and time series of SIMBA snips by rat by group

heatmap_data_rep_10 = get_heatmap_data_by_rat(snips_to_plot, x_array, "replete", "10NaCl")
heatmap_data_rep_45 = get_heatmap_data_by_rat(snips_to_plot, x_array, "replete", "45NaCl")
heatmap_data_dep_10 = get_heatmap_data_by_rat(snips_to_plot, x_array, "deplete", "10NaCl")
heatmap_data_dep_45 = get_heatmap_data_by_rat(snips_to_plot, x_array, "deplete", "45NaCl")

f, ax = plt.subplots(ncols=5, figsize=(7, 1.8),
                     gridspec_kw={"width_ratios": [1, 1, 1, 1, 0.1],
                                  "left": 0.1, "right": 0.95, "wspace": 0.1})

sns.heatmap(heatmap_data_rep_10, ax=ax[0], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=False)
sns.heatmap(heatmap_data_rep_45, ax=ax[1], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=False)
sns.heatmap(heatmap_data_dep_10, ax=ax[2], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=False)
sns.heatmap(heatmap_data_dep_45, ax=ax[3], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=True, cbar_ax=ax[4])



for axis in ax:
    axis.set_xticks([])
    axis.set_yticks([])
    axis.plot((149, 198), (10.5, 10.5), color="black", lw=2, alpha=0.5, clip_on=False)  # Add time scale bar (0.5s at 300Hz)
    axis.plot((49, 148), (-0.5, -0.5), color="black", lw=2, alpha=0.5, clip_on=False)  # Add time scale bar (0.5s at 300Hz)

ax[0].set_yticks([0.5, 9.5], labels=["1", "10"], fontsize=10, rotation=0)
ax[0].set_ylabel("Rats")

if SAVE_FIGS:
    save_figure_atomic(f, "figure_S2_simba_groups_heatmaps", FIGSFOLDER)

In [ ]:
f, ax = plt.subplots(ncols=5, figsize=(7, 1.8), sharey=True,
                     gridspec_kw={"width_ratios": [1, 1, 1, 1, 0.1],
                                  "left": 0.1, "right": 0.95, "bottom": 0.3, "wspace": 0.1})

trial_data_rep_10 = get_trial_data_by_rat(snips_to_plot, x_array, "replete", "10NaCl")
trial_data_rep_45 = get_trial_data_by_rat(snips_to_plot, x_array, "replete", "45NaCl")
trial_data_dep_10 = get_trial_data_by_rat(snips_to_plot, x_array, "deplete", "10NaCl")
trial_data_dep_45 = get_trial_data_by_rat(snips_to_plot, x_array, "deplete", "45NaCl")

alpha=0.2
sigma=1.2

trial_data_smoothed = []
for trial_data in [trial_data_rep_10, trial_data_rep_45, trial_data_dep_10, trial_data_dep_45]:
    trial_data_smoothed.append(gaussian_filter1d(trial_data.T, sigma=sigma))

ax[0].plot(trial_data_smoothed[0], color=colors[0], alpha=alpha)
ax[0].plot(np.nanmean(trial_data_smoothed[0], axis=1), color=colors[0], label="Replete 10NaCl", linewidth=2)

ax[1].plot(trial_data_smoothed[1], color=colors[1], alpha=alpha)
ax[1].plot(np.nanmean(trial_data_smoothed[1], axis=1), color=colors[1], label="Replete 45NaCl", linewidth=2)

ax[2].plot(trial_data_smoothed[2], color=colors[2], alpha=alpha)
ax[2].plot(np.nanmean(trial_data_smoothed[2], axis=1), color=colors[2], label="Deplete 10NaCl", linewidth=2)

ax[3].plot(trial_data_smoothed[3], color=colors[3], alpha=alpha)
ax[3].plot(np.nanmean(trial_data_smoothed[3], axis=1), color=colors[3], label="Deplete 45NaCl", linewidth=2)


for axis in ax[:-1]:  # Skip last axis for colorbar
    axis.set_ylim(-80, 150)
    sns.despine(ax=axis, offset=5)
    axis.set_xlabel("Trial")
    axis.set_xticks([0, 49])
    axis.axhline(0, color="gray", linestyle="--", linewidth=1)

ax[0].set_ylabel("App. Behavior (AUC)", fontsize=10)
ax[0].set_yticks([-50, 0, 50, 100, 150])
ax[4].axis("off")

if SAVE_FIGS:
    save_figure_atomic(f, "figure_S2_simba_groups_time_series", FIGSFOLDER)


In [ ]:
# For figure S2 probably
# ## finding distances between animals

snips_flattened = []
aucs_flattened = []
aucs_smoothed = []
group_index = []
for id in x_array.id.unique():
    x_id = x_array.query("id == @id")
    snips_id = snips_simba[x_id.index]
    x_id_newindex = x_id.reset_index(drop=True)
    for cond in x_id.condition.unique():
        inf = cond
        x_id_cond = x_id_newindex.query("condition == @cond")
        snips_id_cond = snips_id[x_id_cond.index, 50:150]
        snips_flattened.append(snips_id_cond.flatten())
        aucs_flattened.append(np.mean(snips_id_cond, axis=1))
        aucs_smoothed.append(gaussian_filter1d(np.mean(snips_id_cond, axis=1), sigma=1.2))
        group_index.append((id, cond, inf))

aucs_flattened = np.array(aucs_flattened)
aucs_smoothed = np.array(aucs_smoothed)


In [ ]:
sort_idx = sorted(
    range(len(group_index)),
    key=lambda i: (group_index[i][1], group_index[i][2])
)

aucs_smoothed_sorted = [aucs_smoothed[i] for i in sort_idx]
group_index_sorted = [group_index[i] for i in sort_idx]

In [ ]:
metric = "euclidean"
# metric = "correlation"
# metric = "cosine"
# metric = "cityblock"

distances_auc_smoothed = cdist(aucs_smoothed_sorted[:], aucs_smoothed_sorted[:], metric=metric)

In [ ]:
# make fig showing euclidean distances
distance_matric_to_use = distances_auc_smoothed
cmap = plt.get_cmap("Oranges").reversed()

# heatmap with all animals
f, ax = plt.subplots(figsize=(1.8, 1.8))

sns.heatmap(distance_matric_to_use, ax=ax,
            # vmin=15, vmax=45, # for cityblock metric
            vmin=3, vmax=7,
            cmap=cmap,
            cbar=False,
)
ax.set_yticks([])
ax.set_xticks([])

save_figure_atomic(f, "figS1_simba_distance_heatmap_all_animals", folder=FIGSFOLDER)

# heatmap with groups averaged
arr = np.asarray(distance_matric_to_use)
n_groups = 4

if arr.shape != (40, 40):
    raise ValueError(f"Expected a 40x40 matrix, got {arr.shape}")

idx_groups = np.array_split(np.arange(arr.shape[0]), n_groups)
distances_auc_smoothed_4x4 = np.array(
    [[arr[np.ix_(r_idx, c_idx)].mean() for c_idx in idx_groups] for r_idx in idx_groups]
 )


f, ax = plt.subplots(figsize=(1.8, 1.8))
f_, ax_ = plt.subplots(figsize=(0.18, 1.8))
sns.heatmap(distances_auc_smoothed_4x4, ax=ax,
            # vmin=15, vmax=45, # for cityblock metric
            vmin=3, vmax=7,
            cmap=cmap,
            linewidths=1,      # thickness of grid lines
            linecolor="white",
            cbar_ax=ax_
)

ax.set_yticks([])
ax.set_xticks([])

ax_.set_yticks([])

save_figure_atomic(f, "figS1_simba_distance_heatmap", folder=FIGSFOLDER)
save_figure_atomic(f_, "figS1_simba_distance_heatmap_colorbar", folder=FIGSFOLDER)

In [ ]:
mds = MDS(
n_components=2,
metric=True, # use metric MDS
dissimilarity="euclidean",
random_state=42,
n_init=4,
max_iter=300
)

X_scaled = (distance_matric_to_use - np.min(distance_matric_to_use)) / (np.max(distance_matric_to_use) - np.min(distance_matric_to_use))
X_2d = mds.fit_transform(X_scaled) # shape: (n_samples, 2)

In [ ]:
f, ax = plt.subplots(figsize=(1.8, 1.8),
                     gridspec_kw={'left': 0.15, "bottom": 0.15})

reordered_colors = [COLORS[2], COLORS[3], COLORS[0], COLORS[1]]

for i, (id, cond, inf) in enumerate(group_index_sorted[:]):
    if i < 10:
        plt.scatter(X_2d[i, 0], X_2d[i, 1], s=40, label=f"{id}_{cond}", edgecolor=COLORS[2], facecolor='none')
    elif 10 <= i < 20:
        plt.scatter(X_2d[i, 0], X_2d[i, 1], s=40, label=f"{id}_{cond}", edgecolor=COLORS[3], facecolor='none')
    elif 20 <= i < 30:
        plt.scatter(X_2d[i, 0], X_2d[i, 1], s=40, label=f"{id}_{cond}", edgecolor=COLORS[0], facecolor='none')
    elif 30 <= i < 40:
        plt.scatter(X_2d[i, 0], X_2d[i, 1], s=40, label=f"{id}_{cond}", edgecolor=COLORS[1], facecolor='none')

# plot average for each of the four groups
group_means = []
for group in range(4):
    group_indices = range(group*10, (group+1)*10)
    group_mean = X_2d[group_indices].mean(axis=0)
    group_means.append(group_mean)
    ax.scatter(group_mean[0], group_mean[1], s=100, label=f"Group {group+1}", color=reordered_colors[group])

# plot line connecting each dot to its group mean
for i in range(40):
    group = i // 10
    ax.plot([X_2d[i, 0], group_means[group][0]], [X_2d[i, 1], group_means[group][1]], color=reordered_colors[group], alpha=0.3, zorder=0)
    
ax.set_xlabel("MDS1")
ax.set_ylabel("MDS2")

ax.set_xticks([])
ax.set_yticks([])

sns.despine(ax=ax, offset=5)

save_figure_atomic(f, "figS1_simba_distance_mds", folder=FIGSFOLDER)

## Organization

This notebook generates **Figure 1 (Movement Analysis)** only. Each subsequent figure has its own dedicated notebook:

- **figure_1_paper.ipynb**: Movement analysis (current)
- **figure_2_paper.ipynb**: Photometry (dopamine) analysis  
- **figure_3_paper.ipynb**: Neural-behavioral correlation
- **figure_4_paper.ipynb**: Transition analysis
- **figure_5_paper.ipynb**: Cluster analysis

All notebooks share common settings and functions from:
- `src/figure_config.py` — Colors, paths, parameters
- `src/figure_plotting.py` — Data extraction and plotting functions

This keeps each figure focused and manageable, while reducing code duplication.

In [ ]:
# Configuration for Figure Saving
# ───────────────────────────────────────────────────────────────────────
# The SAVE_FIGS setting is loaded from figure_config.py
# Figures are saved in two formats:
#   - PDF for publication (vector format, smaller file size)
#   - PNG for presentations (raster format, high DPI for screen)
#
# All figures follow naming convention:
#   fig{number}_{description}.{pdf|png}
#
# Example:
#   fig1_heatmap_movement_replete.pdf
#   fig1_snips_movement_replete.png
# ───────────────────────────────────────────────────────────────────────

print(f"\nFigure saving is currently: {'ENABLED' if SAVE_FIGS else 'DISABLED'}")
print(f"Figure output folder: {FIGSFOLDER}")
if SAVE_FIGS:
    print("All generated figures will be saved in both PDF and PNG formats.")
else:
    print("To save figures, set SAVE_FIGS = True in src/figure_config.py")

## Figure 1 (Bonus): Movement vs Angular Velocity Comparison

Comparison of head rotation (angular velocity) with body movement to explore the relationship between these metrics across conditions.

In [ ]:
### Per-Rat Time Series: Deplete + 45NaCl (4 metrics across trials)

n_rats = len(rats)
fig, axes = plt.subplots(n_rats, 4, figsize=(14, 2.5*n_rats))

# Handle case where there's only one rat (axes would not be 2D)
if n_rats == 1:
    axes = axes.reshape(1, -1)

# Column labels
col_labels = ['AUC Movement', 'Time Moving', 'AUC Angular Vel', 'Time Above Angvel']

# Add column titles at the top
for col, label in enumerate(col_labels):
    axes[0, col].annotate(label, xy=(0.5, 1.15), xycoords='axes fraction',
                         ha='center', fontsize=11, fontweight='bold')

# Plot each rat
for row, rat in enumerate(rats):
    rat_data = x_dep45[x_dep45['id'] == rat].reset_index(drop=True)
    rat_color = rat_colors[rat]
    trial_nums = range(len(rat_data))
    
    # Column 1: AUC Movement
    axes[row, 0].plot(trial_nums, rat_data['auc_movement'], 'o-', 
                     color=rat_color, markersize=4, linewidth=1, alpha=0.7)
    axes[row, 0].set_ylabel(f'Rat {rat}', fontsize=9, fontweight='bold', labelpad=10)
    axes[row, 0].grid(True, alpha=0.3)
    axes[row, 0].tick_params(labelsize=8)
    if row == n_rats - 1:
        axes[row, 0].set_xlabel('Trial #', fontsize=9)
    
    # Column 2: Time Moving
    axes[row, 1].plot(trial_nums, rat_data['time_moving'], 'o-', 
                     color=rat_color, markersize=4, linewidth=1, alpha=0.7)
    axes[row, 1].set_ylim([0, 1])
    axes[row, 1].grid(True, alpha=0.3)
    axes[row, 1].tick_params(labelsize=8)
    if row == n_rats - 1:
        axes[row, 1].set_xlabel('Trial #', fontsize=9)
    
    # Column 3: AUC Angular Velocity (if available)
    if 'auc_angvel' in rat_data.columns:
        axes[row, 2].plot(trial_nums, rat_data['auc_angvel'], 'o-', 
                         color=rat_color, markersize=4, linewidth=1, alpha=0.7)
        axes[row, 2].grid(True, alpha=0.3)
        axes[row, 2].tick_params(labelsize=8)
        if row == n_rats - 1:
            axes[row, 2].set_xlabel('Trial #', fontsize=9)
    
    # Column 4: Time Above Angular Velocity Threshold (if available)
    if 'time_above_angvel_threshold' in rat_data.columns:
        axes[row, 3].plot(trial_nums, rat_data['time_above_angvel_threshold'], 'o-', 
                         color=rat_color, markersize=4, linewidth=1, alpha=0.7)
        axes[row, 3].set_ylim([0, 1])
        axes[row, 3].grid(True, alpha=0.3)
        axes[row, 3].tick_params(labelsize=8)
        if row == n_rats - 1:
            axes[row, 3].set_xlabel('Trial #', fontsize=9)

fig.suptitle(f'Deplete + 45NaCl: Per-Rat Metrics Across Trials (n={len(x_dep45)} trials)', 
             fontsize=12, fontweight='bold', y=0.995)

plt.tight_layout()
if SAVE_FIGS:
    save_figure_atomic(fig, "fig1_deplete45_perrat_timeseries", FIGSFOLDER)
plt.show()

In [ ]:
### Per-Rat Angular Velocity Threshold Analysis: Deplete + 45NaCl

# Import the threshold function from assemble_all_data
from extract_behav_parameters import smooth_array

# Define different angular velocity thresholds to test
angvel_thresholds = [0.5, 1.0, 1.5, 2.0]
s, e = 50, 150  # Same bin range as used in assembly

# Prepare data: smooth angular velocity snips if not already smoothed
snips_angvel_smooth = smooth_array(snips_angvel, window_size=5)

# Pre-calculate time above threshold for all threshold values and all trials
threshold_data = {}
for thresh in angvel_thresholds:
    time_above = []
    for i in range(snips_angvel_smooth.shape[0]):
        snip = snips_angvel_smooth[i, s:e]
        proportion = len([x for x in snip if x > thresh]) / len(snip)
        time_above.append(proportion)
    threshold_data[thresh] = np.array(time_above)

# Create a dictionary of transition points by rat ID from fits_df
transition_points = {}
if fits_df is not None and len(fits_df) > 0:
    for _, row in fits_df.iterrows():
        transition_points[row['id']] = row['x0_orig']

# Create per-rat figure with one column per threshold
n_rats = len(rats)
fig, axes = plt.subplots(n_rats, len(angvel_thresholds), figsize=(12, 2.5*n_rats))

# Handle case where there's only one rat (axes would not be 2D)
if n_rats == 1:
    axes = axes.reshape(1, -1)

# Column labels with threshold values
col_labels = [f'Threshold = {t}°/frame' for t in angvel_thresholds]

# Add column titles at the top
for col, label in enumerate(col_labels):
    axes[0, col].annotate(label, xy=(0.5, 1.15), xycoords='axes fraction',
                         ha='center', fontsize=10, fontweight='bold')

# Plot each rat across all thresholds
for row, rat in enumerate(rats):
    rat_indices = x_dep45[x_dep45['id'] == rat].index.tolist()
    rat_color = rat_colors[rat]
    trial_nums = range(len(rat_indices))
    
    for col, thresh in enumerate(angvel_thresholds):
        ax = axes[row, col]
        rat_thresh_data = threshold_data[thresh][rat_indices]
        
        ax.plot(trial_nums, rat_thresh_data, 'o-', 
               color=rat_color, markersize=4, linewidth=1, alpha=0.7)
        ax.set_ylim([0, 1])
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=8)
        
        # Add transition point line if available for this rat
        if rat in transition_points:
            x0 = transition_points[rat]
            ax.axvline(x=x0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'x0={x0:.1f}')
            ax.legend(fontsize=7, loc='upper right')
        
        # Add rat label on first column
        if col == 0:
            ax.set_ylabel(f'Rat {rat}', fontsize=9, fontweight='bold', labelpad=10)
        
        # Add x-axis label on bottom row
        if row == n_rats - 1:
            ax.set_xlabel('Trial #', fontsize=9)

fig.suptitle(f'Deplete + 45NaCl: Per-Rat Angular Velocity Threshold Analysis (n={len(x_dep45)} trials)', 
             fontsize=12, fontweight='bold', y=0.995)

plt.tight_layout()
if SAVE_FIGS:
    save_figure_atomic(fig, "fig1_deplete45_perrat_angvel_thresholds", FIGSFOLDER)
plt.show()

print(f"Angular velocity threshold analysis completed.")
print(f"Thresholds tested: {angvel_thresholds} degrees/frame")
print(f"Data window: bins {s}-{e}")
print(f"Number of rats: {n_rats}")
print(f"\nTransition points available for:")
for rat in rats:
    if rat in transition_points:
        print(f"  Rat {rat}: x0_orig = {transition_points[rat]:.2f} trials")
    else:
        print(f"  Rat {rat}: no transition fit available")

In [ ]:
snips_simba_app = snips_simba[x_array.query("condition == 'deplete' and infusiontype == '10NaCl'").index]
sns.heatmap(snips_simba_app, cmap='viridis')

In [ ]:
snips_simba_switch = snips_simba[x_array.query("condition == 'deplete' and infusiontype == '45NaCl'").index]
sns.heatmap(snips_simba_switch, cmap='viridis')

In [ ]:
snips_simba_replete = snips_simba[x_array.query("condition == 'replete'").index]
sns.heatmap(snips_simba_replete, cmap='viridis')

In [ ]:
for id in x_array['id'].unique():
    print(f"Rat {id}:")
    rat_snips = snips_simba[x_array.query("id == @id").index]
    sns.heatmap(rat_snips, cmap='viridis')
    plt.title(f"Rat {id} - SIMBA Snips")
    plt.show()